# Video 08 - Exercise Solutions

Try the exercises in `../exercises.md` first. Spoilers below.

Re-run the Connect + tools + `run_agent` cells from the learner notebook (or this setup cell) before these solutions.


In [ ]:
import json
from datetime import date
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

ORDERS = {42: {"customer": "Alice", "total": 250.0}, 7: {"customer": "Bob", "total": 80.0}}

def get_order_total(customer_id: int):
    order = ORDERS.get(int(customer_id))
    return order["total"] if order else "customer not found"

def calculator(expression: str):
    if not set(expression) <= set("0123456789+-*/(). "):
        return "error: only basic arithmetic is allowed"
    return eval(expression, {"__builtins__": {}})

def get_today():
    return date.today().isoformat()

def tool_spec(name, description, properties=None, required=None):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties or {}, "required": required or []}}}

def add_tool_results(messages, msg, functions):
    messages.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments or "{}")
        func = functions.get(call.function.name)
        result = func(**args) if func else f"error: unknown tool {call.function.name}"
        content = result if isinstance(result, str) else json.dumps(result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

TOOLS = {"get_order_total": get_order_total, "calculator": calculator, "get_today": get_today}
TOOL_SPECS = [
    tool_spec("get_order_total", "Look up the order total in dollars for a customer id.",
              {"customer_id": {"type": "integer"}}, ["customer_id"]),
    tool_spec("calculator", "Evaluate a basic arithmetic expression, e.g. '250 * 0.15'.",
              {"expression": {"type": "string"}}, ["expression"]),
    tool_spec("get_today", "Get today's date in ISO format."),
]

def run_agent(question, max_steps=5, verbose=True, model=MODEL, tools=None, specs=None):
    tools = tools or TOOLS
    specs = specs or TOOL_SPECS
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=model, messages=messages, tools=specs)
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        if verbose:
            for tc in msg.tool_calls:
                print(f"  step {step}: {tc.function.name}({tc.function.arguments})")
        add_tool_results(messages, msg, tools)
    return "Stopped: reached max_steps without a final answer."

print("Setup ready.")


In [ ]:
def apply_discount(total: float, percent: float) -> float:
    return total * (1 - percent / 100)

TOOLS2 = {**TOOLS, "apply_discount": apply_discount}
SPECS2 = TOOL_SPECS + [
    tool_spec("apply_discount", "Apply a percent discount to a total. Returns the discounted amount.",
              {"total": {"type": "number"}, "percent": {"type": "number"}}, ["total", "percent"]),
]

print(run_agent(
    "What's customer 42's order total after a 20% discount?",
    tools=TOOLS2, specs=SPECS2,
))
# Typical path: get_order_total(42) -> apply_discount(250, 20) -> "200"


In [ ]:
print(run_agent("What's 15% of the order total for customer 42?", max_steps=1))
# Returns the "Stopped: reached max_steps..." message (or a partial path).
# With while True and no limit, a confused model could burn tokens forever.


In [ ]:
def run_agent_counting(question, max_steps=5):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    n_calls = 0
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        n_calls += 1
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content, n_calls
        add_tool_results(messages, msg, TOOLS)
    return "Stopped: max_steps", n_calls

for q in ["What is today's date?", "What's 15% of the order total for customer 42?"]:
    answer, n = run_agent_counting(q)
    print(f"calls={n}  Q: {q}")
    print(f"         A: {answer}\n")


In [ ]:
def get_order_total_strict(customer_id: int):
    order = ORDERS.get(int(customer_id))
    if not order:
        raise ValueError("unknown customer")
    return order["total"]

def add_tool_results_safe(messages, msg, functions):
    messages.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments or "{}")
        try:
            result = functions[call.function.name](**args)
        except Exception as e:
            result = f"error: {e}"
        content = result if isinstance(result, str) else json.dumps(result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

def run_agent_safe(question, max_steps=5):
    tools = {"get_order_total": get_order_total_strict, "calculator": calculator, "get_today": get_today}
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        print(f"  step {step}:", [f"{c.function.name}({c.function.arguments})" for c in msg.tool_calls])
        add_tool_results_safe(messages, msg, tools)
    return "Stopped: max_steps"

print(run_agent_safe("What is the order total for customer 999?"))
# Agent should answer politely using the error string - the loop does not crash.
